# Fire Detection YOLOv9 Training

## STEP 1: Install Dependency and Check Environment

In [ ]:
# ============================================================
# Install Dependency
# ============================================================

!pip install -q ultralytics
!pip install -q matplotlib numpy pillow

import os
import sys
import torch
import ultralytics

print("=" * 60)
print("ENVIRONMENT CHECK")
print("=" * 60)

print(f"Python      : {sys.version.split()[0]}")
print(f"Ultralytics : {ultralytics.__version__}")
print(f"PyTorch     : {torch.__version__}")
print(f"CUDA Ready  : {torch.cuda.is_available()}")

# ============================================================
# GPU Information
# ============================================================

if torch.cuda.is_available():
    prop = torch.cuda.get_device_properties(0)

    total = prop.total_memory / 1024**3
    allocated = torch.cuda.memory_allocated(0) / 1024**3
    reserved = torch.cuda.memory_reserved(0) / 1024**3
    free = total - reserved

    print(f"GPU           : {torch.cuda.get_device_name(0)}")
    print(f"VRAM Total    : {total:.2f} GB")
    print(f"VRAM Reserved : {reserved:.2f} GB")
    print(f"VRAM Used     : {allocated:.2f} GB")
    print(f"VRAM Free     : {free:.2f} GB")
else:
    print("WARNING: GPU tidak tersedia.")

# ============================================================
# Detect Kaggle Environment
# ============================================================

print("\n" + "=" * 60)
print("ENVIRONMENT")
print("=" * 60)

if os.path.exists("/kaggle/input") and os.path.exists("/kaggle/working"):
    print("Running inside Kaggle Notebook")
    print("Working Directory :", os.getcwd())
    print("Input Directory   : /kaggle/input")
    print("Output Directory  : /kaggle/working")
else:
    print("Not running inside Kaggle Notebook")

## STEP 2: Import Library

In [ ]:
# Import all necessary libraries
import os
import sys
import glob
import time
import zipfile
from pathlib import Path

import torch
import numpy as np
from ultralytics import YOLO
from PIL import Image
import matplotlib.pyplot as plt

print("All libraries imported successfully")

### STEP 3: Check GPU

In [ ]:
# Detailed GPU information
if torch.cuda.is_available():
    print("="*60)
    print("DETAILED GPU INFORMATION")
    print("="*60)
    
    device = torch.cuda.current_device()
    props = torch.cuda.get_device_properties(device)
    
    print(f"Device Index: {device}")
    print(f"Device Name: {props.name}")
    print(f"Total Memory: {props.total_memory / 1024**3:.2f} GB")
    print(f"Multi-processor Count: {props.multi_processor_count}")
    print(f"Compute Capability: {props.major}.{props.minor}")
    print(f"Max Threads per Block: {props.max_threads_per_block}")
    if hasattr(props, "max_grid_size"):
        print(f"Max Grid Size: {props.max_grid_size}")
    if hasattr(props, "max_threads_dim"):
        print(f"Max Block Size: {props.max_threads_dim}")
    
    # Memory info
    allocated = torch.cuda.memory_allocated(device) / 1024**3
    reserved = torch.cuda.memory_reserved(device) / 1024**3
    total = props.total_memory / 1024**3
    
    print(f"\nMemory Status:")
    print(f"   - Allocated: {allocated:.2f} GB")
    print(f"   - Reserved: {reserved:.2f} GB")
    print(f"   - Total: {total:.2f} GB")
    print(f"   - Free: {total - allocated:.2f} GB")
    
    print("\n" + "="*60)
    print("GPU ready for training")
    print("="*60)
else:
    print("WARNING: No GPU detected!")
    print("Training will use CPU - this will be very slow!")
    print("Consider enabling GPU in Kaggle notebook settings.")

## STEP 4: Load Dataset

In [ ]:
# Kaggle dataset path - using Kaggle Input
# Dataset should be uploaded to Kaggle Input with name 'fire-detection-dataset'
# Structure: /kaggle/input/fire-detection-dataset/data.yaml

# Set dataset path for Kaggle
DATASET_PATH = "/kaggle/input/datasets/hendracode/fire-detection-yolov9-dataset/DATASET/data.yaml"

# Alternative path if dataset is in working directory
# DATASET_PATH = "/kaggle/working/data.yaml"

print("="*60)
print("LOADING DATASET")
print("="*60)
print(f"Dataset path: {DATASET_PATH}")

# Check if dataset exists
if os.path.exists(DATASET_PATH):
    print(f"Dataset ditemukan di: {DATASET_PATH}")
    
    # Read and display dataset configuration
    with open(DATASET_PATH, 'r') as f:
        dataset_config = f.read()
    
    print("\nDataset Configuration (data.yaml):")
    print("-" * 40)
    print(dataset_config)
    print("-" * 40)
    
    # Extract dataset directory from path
    dataset_dir = os.path.dirname(DATASET_PATH)
    print(f"\nDataset directory: {dataset_dir}")
    
    # List dataset structure
    print(f"\nDataset structure:")
    for root, dirs, files in os.walk(dataset_dir):
        level = root.replace(dataset_dir, '').count(os.sep)
        indent = ' ' * 2 * level
        print(f'{indent}{os.path.basename(root)}/')
        subindent = ' ' * 2 * (level + 1)
        for file in files[:10]:  # Show first 10 files
            print(f'{subindent}{file}')
        if len(files) > 10:
            print(f'{subindent}... dan {len(files) - 10} file lainnya')
else:
    print(f"Dataset TIDAK ditemukan di: {DATASET_PATH}")
    print("\nSolusi:")
    print("1. Pastikan dataset sudah diupload ke Kaggle Input")
    print("2. Pastikan nama dataset di Kaggle Input adalah: 'fire-detection-dataset'")
    print("3. Pastikan file data.yaml ada di root dataset")
    print("4. Atau ubah DATASET_PATH di cell ini sesuai lokasi dataset Anda")
    
    # Try alternative paths
    alt_paths = [
        "/kaggle/input/fire-detection/data.yaml",
        "/kaggle/input/fire_detection/data.yaml",
        "/kaggle/working/data.yaml"
    ]
    
    print("\nMencoba path alternatif...")
    for alt_path in alt_paths:
        if os.path.exists(alt_path):
            print(f"Dataset ditemukan di: {alt_path}")
            DATASET_PATH = alt_path
            break
    else:
        print("Dataset tidak ditemukan di path manapun")

## STEP 5: Training Configuration

In [ ]:
# Training configuration - all parameters in one cell for easy modification
print("="*60)
print("TRAINING CONFIGURATION")
print("="*60)

# ===== MODEL CONFIGURATION =====
MODEL = "yolov9c.pt"  # Pretrained YOLOv9 model
print(f"Model: {MODEL}")

# ===== DATASET CONFIGURATION =====
# Use the dataset path from previous step
print(f"Dataset: {DATASET_PATH}")

# ===== TRAINING PARAMETERS =====
EPOCHS = 50                    # Number of training epochs
BATCH_SIZE = 16               # Batch size (16 optimal for Kaggle GPU)
IMAGE_SIZE = 640              # Input image size
DEVICE = "0" if torch.cuda.is_available() else "cpu"  # Use GPU if available
PATIENCE = 20                 # Early stopping patience
WORKERS = 2                  # Number of data loader workers
SAVE_PERIOD = 10             # Save checkpoint every N epochs

# ===== PROJECT CONFIGURATION =====
PROJECT_NAME = "fire_detection_yolov9_kaggle"  # Project name for runs directory
RUN_NAME = "run_001"         # Run name
VERBOSE = True               # Verbose training output

# ===== OUTPUT CONFIGURATION =====
OUTPUT_DIR = "/kaggle/working/"  # All outputs will be saved here
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Epochs: {EPOCHS}")
print(f"Batch Size: {BATCH_SIZE}")
print(f"Image Size: {IMAGE_SIZE}")
print(f"Device: {'GPU' if DEVICE == '0' else 'CPU'}")
print(f"Patience: {PATIENCE}")
print(f"Workers: {WORKERS}")
print(f"Save Period: {SAVE_PERIOD}")
print(f"Project: {PROJECT_NAME}")
print(f"Run: {RUN_NAME}")
print(f"Output: {OUTPUT_DIR}")

print("\n" + "="*60)
print("Configuration ready")
print("="*60)

## STEP 6: Start Training

In [ ]:
# Start training with auto-resume functionality
print("="*60)
print("STARTING TRAINING")
print("="*60)

# Check for existing checkpoint
checkpoint_path = os.path.join(OUTPUT_DIR, PROJECT_NAME, RUN_NAME, "weights", "last.pt")
resume_training = os.path.exists(checkpoint_path)

if resume_training:
    print(f"Checkpoint ditemukan: {checkpoint_path}")
    print("Training akan dilanjutkan (resume=True)")
else:
    print("Tidak ada checkpoint, training dimulai dari pretrained model")
    print("Loading pretrained model...")

# Load model
model = YOLO(MODEL)

# Training start time
training_start_time = time.time()
print(f"Training dimulai: {time.ctime(training_start_time)}")

# Start training
print("\nStarting training...")
print("Training akan memakan waktu 2-4 jam")
print("Training otomatis akan menyimpan checkpoint setiap {SAVE_PERIOD} epoch")
print("Jika runtime berhenti, jalankan cell ini lagi untuk melanjutkan")

try:
    results = model.train(
        data=DATASET_PATH,
        epochs=EPOCHS,
        imgsz=IMAGE_SIZE,
        batch=BATCH_SIZE,
        name=RUN_NAME,
        patience=PATIENCE,
        device=DEVICE,
        workers=WORKERS,
        verbose=VERBOSE,
        plots=True,
        resume=resume_training,
        project=OUTPUT_DIR + PROJECT_NAME,
        exist_ok=True,
        save_period=SAVE_PERIOD,
    )
    
    # Training completed successfully
    training_end_time = time.time()
    training_duration = training_end_time - training_start_time
    
    print("\n" + "="*60)
    print("TRAINING SELESAI!")
    print("="*60)
    print(f"Training berakhir: {time.ctime(training_end_time)}")
    print(f"Durasi training: {training_duration/3600:.2f} jam")
    
    # Get output directory
    output_run_dir = os.path.join(OUTPUT_DIR, PROJECT_NAME, RUN_NAME)
    print(f"Hasil training tersimpan di: {output_run_dir}")
    
    # List output files
    print(f"\nFile-file output:")
    for root, dirs, files in os.walk(output_run_dir):
        level = root.replace(output_run_dir, '').count(os.sep)
        indent = ' ' * 2 * level
        print(f'{indent}{os.path.basename(root)}/')
        subindent = ' ' * 2 * (level + 1)
        for file in files[:15]:  # Show first 15 files
            print(f'{subindent}{file}')
        if len(files) > 15:
            print(f'{subindent}... dan {len(files) - 15} file lainnya')
except Exception as e:
    print(f"\nERROR saat training: {str(e)}")
    print("\nTips:")
    print("- Cek apakah GPU tersedia")
    print("- Kurangi batch size jika memory habis")
    print("- Cek dataset path")
    raise

## STEP 7: Evaluate Model

In [ ]:
print("=" * 60)
print("MODEL EVALUATION")
print("=" * 60)

# Best model path
output_run_dir = os.path.join(OUTPUT_DIR, PROJECT_NAME, RUN_NAME)
best_pt_path = os.path.join(output_run_dir, "weights", "best.pt")

if os.path.exists(best_pt_path):

    print(f"Loading best model: {best_pt_path}")

    # Load model
    model = YOLO(best_pt_path)

    print("Running validation...")

    eval_start_time = time.time()

    results = model.val(
        data=DATASET_PATH,
        imgsz=IMAGE_SIZE,
        batch=BATCH_SIZE,
        device=DEVICE,
        split="test",
        plots=True,
    )

    eval_end_time = time.time()

    # ============================================================
    # Calculate Metrics
    # ============================================================

    map50 = float(results.box.map50)
    map5095 = float(results.box.map)

    precision = float(np.mean(results.box.p))
    recall = float(np.mean(results.box.r))

    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0.0
    )

    # ============================================================
    # Display Results
    # ============================================================

    print("\n" + "=" * 60)
    print("VALIDATION SELESAI!")
    print("=" * 60)

    print(f"Durasi validasi : {eval_end_time - eval_start_time:.2f} detik")

    print("\nEVALUATION METRICS")
    print("-" * 60)
    print(f"mAP50        : {map50:.4f}")
    print(f"mAP50-95     : {map5095:.4f}")
    print(f"Precision    : {precision:.4f}")
    print(f"Recall       : {recall:.4f}")
    print(f"F1-Score     : {f1:.4f}")

    # ============================================================
    # Performance Assessment
    # ============================================================

    print("\nMODEL ASSESSMENT")

    if map50 >= 0.90:
        print("EXCELLENT! Model memiliki performa yang sangat baik.")
    elif map50 >= 0.80:
        print("VERY GOOD! Model memiliki performa yang baik.")
    elif map50 >= 0.70:
        print("GOOD! Model layak digunakan.")
    else:
        print("Model masih perlu ditingkatkan dengan training tambahan.")

    # ============================================================
    # Save Metrics
    # ============================================================

    metrics_file = os.path.join(
        output_run_dir,
        "evaluation_metrics.txt"
    )

    with open(metrics_file, "w") as f:

        f.write("YOLOv9 Fire Detection Evaluation\n")
        f.write("=" * 60 + "\n\n")

        f.write(f"Evaluation Time : {time.ctime()}\n\n")

        f.write(f"mAP50        : {map50:.4f}\n")
        f.write(f"mAP50-95     : {map5095:.4f}\n")
        f.write(f"Precision    : {precision:.4f}\n")
        f.write(f"Recall       : {recall:.4f}\n")
        f.write(f"F1-Score     : {f1:.4f}\n")

    print(f"\nMetrics berhasil disimpan:")
    print(metrics_file)

    # ============================================================
    # Store Results
    # ============================================================

    evaluation_results = {
        "map50": map50,
        "map50_95": map5095,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "evaluation_time": eval_end_time - eval_start_time,
        "best_model": best_pt_path,
        "metrics_file": metrics_file,
    }

else:

    print(f"Best model tidak ditemukan:\n{best_pt_path}")
    print("Silakan jalankan proses training terlebih dahulu.")

    evaluation_results = None

## STEP 8: Export Model

In [ ]:
# Export model to various formats
print("="*60)
print("EXPORTING MODEL")
print("="*60)

# Path to best model
output_run_dir = os.path.join(OUTPUT_DIR, PROJECT_NAME, RUN_NAME)
best_pt_path = os.path.join(output_run_dir, "weights", "best.pt")

if os.path.exists(best_pt_path):
    print(f"Loading model: {best_pt_path}")
    model = YOLO(best_pt_path)
    
    # Create export directory
    export_dir = os.path.join(output_run_dir, "export")
    os.makedirs(export_dir, exist_ok=True)
    
    # Export to TorchScript
    print("\nExporting to TorchScript...")
    try:
        torchscript_path = os.path.join(export_dir, "best_torchscript.pt")
        model.export(format="torchscript", imgsz=IMAGE_SIZE)
        # Move to export directory
        if os.path.exists("best_torchscript.pt"):
            os.rename("best_torchscript.pt", torchscript_path)
        print(f"TorchScript model tersimpan di: {torchscript_path}")
    except Exception as e:
        print(f"Failed to export TorchScript: {str(e)}")
    
    # Export to ONNX
    print("\nExporting to ONNX...")
    try:
        onnx_path = os.path.join(export_dir, "best.onnx")
        model.export(format="onnx", imgsz=IMAGE_SIZE)
        # Move to export directory
        if os.path.exists("best.onnx"):
            os.rename("best.onnx", onnx_path)
        print(f"ONNX model tersimpan di: {onnx_path}")
    except Exception as e:
        print(f"Failed to export ONNX: {str(e)}")
    
    # Export to TensorRT (if supported)
    print("\nChecking TensorRT support...")
    try:
        # Check if TensorRT is available
        try:
            import tensorrt
            tensorrt_available = True
        except ImportError:
            tensorrt_available = False
        
        if tensorrt_available:
            print("TensorRT available, exporting...")
            engine_path = os.path.join(export_dir, "best.engine")
            model.export(format="engine", imgsz=IMAGE_SIZE)
            if os.path.exists("best.engine"):
                os.rename("best.engine", engine_path)
            print(f"TensorRT engine tersimpan di: {engine_path}")
        else:
            print("TensorRT not available in Kaggle environment")
    except Exception as e:
        print(f"Failed to export TensorRT: {str(e)}")
    
    print("\n" + "="*60)
    print("EXPORT SELESAI!")
    print("="*60)
    
    # List exported files
    print(f"\nFile-file export:")
    for file in os.listdir(export_dir):
        file_path = os.path.join(export_dir, file)
        file_size = os.path.getsize(file_path) / 1024**2  # MB
        print(f"   - {file} ({file_size:.2f} MB)")
else:
    print(f"Model tidak ditemukan di: {best_pt_path}")
    print("Jalankan training terlebih dahulu!")

## STEP 9: Compress Output (ZIP)

In [ ]:
print("=" * 60)
print("COMPRESSING OUTPUT")
print("=" * 60)

output_run_dir = os.path.join(OUTPUT_DIR, PROJECT_NAME, RUN_NAME)

if os.path.exists(output_run_dir):

    timestamp = time.strftime("%Y%m%d_%H%M%S")

    zip_filename = f"fire_detection_yolov9_results_{timestamp}.zip"
    zip_path = os.path.join(OUTPUT_DIR, zip_filename)

    print(f"Creating ZIP file...")
    print(f"Source : {output_run_dir}")
    print(f"Output : {zip_path}")

    files_to_include = [

        # Model
        "weights/best.pt",
        "weights/last.pt",

        # Metrics
        "results.csv",
        "evaluation_metrics.txt",
        "args.yaml",

        # Curves
        "BoxPR_curve.png",
        "BoxP_curve.png",
        "BoxR_curve.png",
        "BoxF1_curve.png",

        # Confusion Matrix
        "confusion_matrix.png",
        "confusion_matrix_normalized.png",

        # Summary
        "results.png",
        "labels.jpg",

        # Sample predictions
        "train_batch0.jpg",
        "train_batch1.jpg",
        "train_batch2.jpg",

        "val_batch0_labels.jpg",
        "val_batch0_pred.jpg",

        "val_batch1_labels.jpg",
        "val_batch1_pred.jpg",

        "val_batch2_labels.jpg",
        "val_batch2_pred.jpg",
    ]

    added = 0

    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:

        for file in files_to_include:

            file_path = os.path.join(output_run_dir, file)

            if os.path.exists(file_path):

                arcname = os.path.relpath(file_path, OUTPUT_DIR)

                zipf.write(file_path, arcname)

                print(f"✓ {arcname}")

                added += 1

            else:

                print(f"⚠ Tidak ditemukan : {file}")

    print("\n" + "=" * 60)
    print("ZIP BERHASIL DIBUAT")
    print("=" * 60)

    print(f"Jumlah file : {added}")
    print(f"Lokasi      : {zip_path}")
    print(f"Ukuran      : {os.path.getsize(zip_path)/1024**2:.2f} MB")

    main_zip_path = zip_path

else:

    print("Folder output tidak ditemukan.")
    print("Training belum dijalankan.")

    main_zip_path = None